# CAPA Notebook

This notebook provides a working implementation for the Control-System Camera AI-Powered Productivity Assistant (CAPA). It includes a camera-based focus detector and real-time AI analysis using a local LLM API running at `http://127.0.0.1:1234`.


## Setup

Before running this notebook, ensure the local LLM API is running at `http://127.0.0.1:1234`.

Install the required Python packages, then run the next cells. The notebook will test the API connection and report success or failure.


In [ ]:
!pip install transformers torch accelerate opencv-python numpy requests

## Load the LLM and define helper functions

This cell connects to the local LLM API running at `http://127.0.0.1:1234` and creates helper functions for generating productivity messages and session summaries. The API should be running before executing this cell.


In [ ]:
import requests
import json

# Local LLM API endpoint
LOCAL_LLM_URL = "http://127.0.0.1:1234"
API_ENDPOINT = f"{LOCAL_LLM_URL}/v1/chat/completions"

def test_local_api():
    """Test connection to local LLM API"""
    try:
        response = requests.post(
            API_ENDPOINT,
            json={
                "model": "local",
                "messages": [{"role": "user", "content": "test"}],
                "max_tokens": 10,
                "temperature": 0.7
            },
            timeout=5
        )
        if response.status_code == 200:
            print("✓ Local LLM API connected successfully!")
            return True
        else:
            print(f"✗ API returned status {response.status_code}")
            return False
    except Exception as e:
        print(f"✗ Failed to connect to local LLM API: {e}")
        return False

def generate_productivity_message(prompt: str, max_new_tokens: int = 128):
    """Generate productivity message using local LLM API"""
    try:
        response = requests.post(
            API_ENDPOINT,
            json={
                "model": "local",
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": max_new_tokens,
                "temperature": 0.8,
                "top_p": 0.92,
            },
            timeout=30
        )
        
        if response.status_code == 200:
            data = response.json()
            # Handle OpenAI-compatible format
            if "choices" in data and len(data["choices"]) > 0:
                return data["choices"][0]["message"]["content"].strip()
        
        return "Unable to generate response from local LLM."
    except Exception as e:
        return f"Error connecting to LLM: {str(e)}"

def summarize_session(events):
    """Generate session summary using local LLM API"""
    prompt_text = "You are an AI productivity coach. "
    prompt_text += "The user session data below describes desk presence, motion, and focus events. "
    prompt_text += "Create a concise summary outlining when the user was focused, when they were distracted, and one recommendation for improving the next session.\n\nSession events:\n"
    
    for event in events:
        prompt_text += f"- {event['timestamp']}: state={event['state']}, motion={event['motion']:.0f}\n"
    
    prompt_text += "\nSummary:"
    
    return generate_productivity_message(prompt_text, max_new_tokens=180)

print("Testing connection to local LLM API...")
api_connected = test_local_api()

if api_connected:
    print("✓ LLM helper functions configured for local API at 127.0.0.1:1234")
else:
    print("⚠ Warning: Local LLM API not found. Ensure it's running at http://127.0.0.1:1234")


## Enhanced Camera Monitoring with Procrastination Detection

This enhanced version includes:
- **Real-time camera display** in a separate OpenCV window showing motion detection
- **Live transcript** logged directly in the notebook showing focus/quiet states
- **Procrastination detection** that identifies when motion is too low for extended periods
- **AI-powered suggestions** that provide personalized productivity tips when procrastination is detected
- **Focus analysis** with percentage calculations and recommendations at the end of the session

## Session States and Indicators

- **✓ FOCUSED**: Motion detected above threshold - user is actively engaged
- **⚠ QUIET**: Motion below threshold - user may be reading, thinking, or distracted
- **[ALERT]**: Procrastination detected - sustained low motion triggers AI suggestion
- **Camera Window**: Green border = focused, Orange border = quiet; Press 'Q' to exit

In [ ]:
import cv2
import numpy as np
import time
from collections import deque
from IPython.display import clear_output, display
import sys

def detect_procrastination(motion_history, threshold_low=500, window_size=10):
    """
    Detect procrastination based on sustained low motion.
    Returns True if the user has been idle for too long.
    """
    if len(motion_history) < window_size:
        return False
    recent = list(motion_history)[-window_size:]
    avg_motion = np.mean(recent)
    return avg_motion < threshold_low

def get_procrastination_suggestion(motion_level, idle_duration):
    """
    Generate AI-powered suggestions when procrastination is detected.
    """
    if idle_duration < 5:
        return None
    
    prompt = f"The user has been idle for {idle_duration} seconds with very low motion ({motion_level:.0f}). Generate a brief, encouraging productivity tip (1-2 sentences) to help them refocus."
    
    try:
        suggestion = generate_productivity_message(prompt, max_new_tokens=64)
        return suggestion
    except:
        return "Time to refocus! Take a deep breath and get back to your task."

def run_focus_session_enhanced(duration_seconds=60, min_motion_area=1500, display_interval=5):
    """
    Enhanced focus session with:
    - Real-time camera display in separate window
    - Session transcript logged to notebook
    - Procrastination detection with AI suggestions
    """
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("Unable to open the default webcam.")

    ret, frame = cap.read()
    if not ret:
        cap.release()
        raise RuntimeError("Unable to read from the webcam.")

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    background = cv2.GaussianBlur(gray, (21, 21), 0)
    events = []
    motion_history = deque(maxlen=30)
    idle_counter = 0
    last_display_time = time.time()
    start_time = time.time()
    
    transcript = []
    
    print("=" * 60)
    print("CAPA FOCUS SESSION STARTED")
    print("=" * 60)
    print(f"Duration: {duration_seconds} seconds")
    print(f"Motion threshold: {min_motion_area}")
    print("\nReal-time Transcript:")
    print("-" * 60)

    while time.time() - start_time < duration_seconds:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (21, 21), 0)
        diff = cv2.absdiff(background, blur)
        thresh = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)[1]
        thresh = cv2.dilate(thresh, None, iterations=2)

        contours, _ = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        motion_area = sum(cv2.contourArea(c) for c in contours)
        state = "focused" if motion_area > min_motion_area else "quiet"
        
        motion_history.append(motion_area)

        timestamp = time.strftime("%H:%M:%S", time.localtime())
        elapsed = int(time.time() - start_time)
        
        event = {
            "timestamp": timestamp,
            "elapsed": elapsed,
            "motion": float(motion_area),
            "state": state,
        }
        events.append(event)

        # Display on frame
        text = f"State: {state} | Motion: {motion_area:.0f} | Time: {elapsed}s"
        cv2.putText(frame, text, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        
        # Add border color based on state
        color = (0, 255, 0) if state == "focused" else (0, 165, 255)
        cv2.rectangle(frame, (10, 10), (frame.shape[1]-10, frame.shape[0]-10), color, 3)
        
        cv2.imshow("CAPA Focus Monitor", frame)

        # Check for procrastination
        is_procrastinating = detect_procrastination(motion_history)
        
        if is_procrastinating:
            idle_counter += 1
            if idle_counter % 10 == 0:
                suggestion = get_procrastination_suggestion(motion_area, idle_counter)
                if suggestion:
                    msg = f"[ALERT {elapsed}s] Procrastination detected! Suggestion: {suggestion}"
                    transcript.append(msg)
                    print(msg)
        else:
            idle_counter = 0

        # Log session events periodically
        if time.time() - last_display_time >= display_interval:
            if events[-1]["state"] == "focused":
                log_msg = f"[{timestamp}] ✓ User is FOCUSED (motion: {motion_area:.0f})"
            else:
                log_msg = f"[{timestamp}] ⚠ User is QUIET (motion: {motion_area:.0f})"
            transcript.append(log_msg)
            print(log_msg)
            last_display_time = time.time()

        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("\n[Session] Terminated by user (Q key pressed)")
            break

    cap.release()
    cv2.destroyAllWindows()
    
    print("-" * 60)
    print("CAPA FOCUS SESSION ENDED")
    print("=" * 60)
    
    return events, transcript

print("Enhanced camera focus session helper defined. Run `run_focus_session_enhanced()` to start monitoring.")


## Run a Focus Session and Generate a Summary

Start the monitoring session and generate an AI summary from the collected events.

In [ ]:
session_events, transcript = run_focus_session_enhanced(duration_seconds=60, min_motion_area=1500)

print("\n\nSESSION SUMMARY")
print("=" * 60)
print(f"Total events captured: {len(session_events)}")
print(f"Procrastination alerts: {sum(1 for t in transcript if 'ALERT' in t)}")

# Analyze focus patterns
focused_count = sum(1 for e in session_events if e["state"] == "focused")
quiet_count = len(session_events) - focused_count
focus_percentage = (focused_count / len(session_events) * 100) if session_events else 0

print(f"Focus time: {focus_percentage:.1f}%")
print(f"Quiet time: {100 - focus_percentage:.1f}%")

print("\nGenerating AI session analysis...")
summary = summarize_session(session_events)
print("\nAI-GENERATED SESSION SUMMARY:")
print("-" * 60)
print(summary)
print("-" * 60)

# Display final recommendation
if focus_percentage < 50:
    recommendation = "Your focus level was below average. Consider using the CAPA tool more frequently or adjusting your work environment to minimize distractions."
elif focus_percentage < 75:
    recommendation = "Good work! Your focus was above average. Keep maintaining consistent attention breaks and stay hydrated."
else:
    recommendation = "Excellent focus session! You maintained high concentration throughout. Keep up this momentum!"

print(f"\nRECOMMENDATION: {recommendation}")
